# 10 — Typing, Contracts, and Protocols

Python type hints are primarily for readers, editors, and static checkers. Learn how to express collection types, optional results, structured mappings, and behavior-based interfaces without confusing hints with runtime validation.

*Intermediate · 75–120 min*

## You will learn

- Annotate functions, containers, and optional return types
- Use type aliases and TypedDict for structured records
- Understand Protocol and structural interfaces
- Separate static hints from runtime input checking

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Type annotations and unions

Type hints document expected values and help a static checker find mistakes before execution. In Python 3.10+, use list[int], dict[str, float], and str | None. Hints do not automatically reject incorrect runtime arguments.

In [ ]:
def average(values: list[float]) -> float | None:
    """Average values, or return None for an empty input."""
    return sum(values) / len(values) if values else None

scores: list[float] = [82.0, 94.0, 76.0]
print(average(scores))
print(average([]))
print(average.__annotations__)

**Think about it:** What would actually happen if a caller passed a list of strings?

### 2. Type aliases and TypedDict

TypedDict describes a mapping with expected key names and value types. It is useful for JSON-like data in application boundaries. At runtime, a TypedDict is still an ordinary dictionary; validation must be explicit.

In [ ]:
from typing import TypedDict

class Student(TypedDict):
    name: str
    score: float

def display_student(student: Student) -> str:
    return f"{student['name']}: {student['score']:.1f}"

record: Student = {"name": "Ada", "score": 90.5}
print(display_student(record))
print(type(record).__name__)

**Think about it:** Why does a TypedDict not guarantee incoming JSON has valid fields?

### 3. Protocols and structural subtyping

A Protocol defines operations expected from an object, even if the class does not inherit from the Protocol. This documents duck typing. Static checkers verify compatibility, but Python does not generally enforce Protocol conformance automatically.

In [ ]:
from typing import Protocol

class Renderable(Protocol):
    def render(self) -> str: ...

class Heading:
    def __init__(self, text: str):
        self.text = text
    def render(self) -> str:
        return f"# {self.text}"

def show(item: Renderable) -> str:
    return item.render()

print(show(Heading("Introduction")))

**Think about it:** Which other object could be passed to show without inheriting from Renderable?

### 4. Callable and generic collection signatures

Callable expresses the argument and result types of functions passed to other functions. Be precise enough to be useful but avoid making introductory APIs overly generic. Editors can use these hints for suggestions and static error detection.

In [ ]:
from collections.abc import Callable, Iterable

def transform(values: Iterable[int], operation: Callable[[int], int]) -> list[int]:
    return [operation(value) for value in values]

def twice(number: int) -> int:
    return 2 * number

print(transform([2, 4, 6], twice))
print(transform(range(3), lambda x: x * x))

**Think about it:** How does this differ from hardcoding one operation into transform?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_data_quality_and_testing_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/10', {1: '992e8d1e664a3ed3', 2: '979f0030714a7248', 3: 'e3e7d84363c04fec', 4: 'b131748ae9de1617'})

**Exercise 1.** Annotate a lookup that may return `None` with `Optional`.

In [ ]:
from typing import Optional, get_type_hints

def find_student(students: dict[str, int], name: str) -> ___:
    return students.get(name)

hints = get_type_hints(find_student)
check(1, (find_student({"ana": 90}, "ana"), find_student({}, "x"), str(hints["return"])))

**Exercise 2.** A `TypedDict` describing a product from JSON.

In [ ]:
from typing import TypedDict

class Product(___):
    sku: str
    price: float

item: Product = {"sku": "A1", "price": 2.5}
check(2, (sorted(Product.__annotations__), item["price"]))

**Exercise 3.** A `Protocol` for anything with `save(data)`; structural typing needs no inheritance.

In [ ]:
from typing import Protocol

class Saver(Protocol):
    def save(self, data: str) -> None: ...

class MemorySaver:              # does NOT inherit from Saver
    def __init__(self): self.items = []
    def save(self, data: str) -> None: self.items.append(data)

def store(saver: Saver, data: str) -> None:
    ___

m = MemorySaver()
store(m, "x"); store(m, "y")
check(3, m.items)

**Exercise 4.** Accept a transformation function typed as `Callable`.

In [ ]:
from typing import Callable

def apply(values: list[int], transform: ___) -> list[int]:
    return [transform(v) for v in values]

check(4, apply([1, 2, 3], ___))

## Common mistakes

- Annotations guide static tools, not automatic runtime validation.
- TypedDict objects are plain dictionaries when code executes.
- Use explicit validation at untrusted I/O boundaries.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Do type hints stop wrong types at runtime?

   - **a)** No, they are checked by tools, not enforced by Python
   - **b)** Only for int
   - **c)** Yes, always

**2.** A Protocol describes…

   - **a)** A required base class
   - **b)** A network protocol
   - **c)** The methods an object must have (structural typing)

**3.** Optional[int] means…

   - **a)** Any type
   - **b)** An int with a default
   - **c)** int or None

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/10', {1: '83f2c72b3568', 2: '416f48af5d4c', 3: 'b7f3ef96236d'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [11 — Logging, Configuration, and Error Boundaries](11_logging_configuration_and_errors.ipynb)